# STAGNet – Protocol II: MediaPipe FaceMesh Landmark Extraction

## Overview

| | |
|---|---|
| **Notebook purpose** | Extract facial landmark coordinates from BIWI RGB frames using MediaPipe FaceMesh and store the results alongside the associated head-pose labels. |
| **Landmark detector** | MediaPipe FaceMesh |
| **Evaluation protocol** | Protocol II – subject-independent BIWI split (16 training sequences / 8 testing sequences) |
| **Expected input** | BIWI RGB frames with corresponding yaw / pitch / roll annotation files, accessed through Google Drive. |
| **Expected output** | Per-sequence landmark files containing landmark coordinates and pose labels, saved to the specified Google Drive output directory. |

> **Before running this notebook**, update `DATASET_ROOT` and `OUTPUT_ROOT` in the configuration cell below to match the paths in your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---------------------------------------------------------------
# Configuration – update these paths before running the notebook
# ---------------------------------------------------------------
DATASET_ROOT = "/content/drive/MyDrive/STAGNet_Datasets/BIWI"
OUTPUT_ROOT  = "/content/drive/MyDrive/STAGNet_Outputs/Protocol_II"

# Protocol II sequence split
TRAIN_SEQUENCES = [
    "01", "02", "03", "04", "05", "06", "07", "08",
    "09", "10", "11", "12", "13", "14", "15", "16"
]
TEST_SEQUENCES = ["17", "18", "19", "20", "21", "22", "23", "24"]

In [ ]:
# Install dependencies
!pip install mediapipe opencv-python-headless

In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh

print("Libraries loaded successfully.")

In [ ]:
def extract_landmarks_facemesh(image_bgr, face_mesh):
    """Return normalised (x, y, z) landmarks for the first detected face, or None."""
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(image_rgb)
    if results.multi_face_landmarks:
        lm = results.multi_face_landmarks[0].landmark
        coords = np.array([[p.x, p.y, p.z] for p in lm], dtype=np.float32)
        return coords
    return None


def process_split(sequences, split_name, dataset_root, output_root):
    """Process a list of BIWI sequences and save extracted landmarks."""
    os.makedirs(os.path.join(output_root, split_name), exist_ok=True)

    with mp_face_mesh.FaceMesh(
        static_image_mode=True,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5
    ) as face_mesh:

        for seq in sequences:
            seq_dir = os.path.join(dataset_root, seq)
            if not os.path.isdir(seq_dir):
                print(f"[WARN] Sequence directory not found: {seq_dir}")
                continue

            landmarks_list, poses_list, frame_ids = [], [], []

            for fname in sorted(os.listdir(seq_dir)):
                if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    continue

                img_path = os.path.join(seq_dir, fname)
                pose_path = os.path.splitext(img_path)[0] + "_pose.txt"

                if not os.path.isfile(pose_path):
                    continue

                image = cv2.imread(img_path)
                if image is None:
                    continue

                image_resized = cv2.resize(image, (224, 224))
                landmarks = extract_landmarks_facemesh(image_resized, face_mesh)

                pose = np.loadtxt(pose_path)  # expected: [yaw, pitch, roll]

                landmarks_list.append(landmarks if landmarks is not None
                                      else np.zeros((478, 3), dtype=np.float32))
                poses_list.append(pose)
                frame_ids.append(os.path.splitext(fname)[0])

            out_file = os.path.join(output_root, split_name, f"seq_{seq}.npz")
            np.savez(
                out_file,
                landmarks=np.array(landmarks_list, dtype=np.float32),
                poses=np.array(poses_list, dtype=np.float32),
                frame_ids=np.array(frame_ids)
            )
            print(f"[{split_name}] Sequence {seq}: {len(frame_ids)} frames → {out_file}")

In [ ]:
print("Processing training split ...")
process_split(TRAIN_SEQUENCES, "train", DATASET_ROOT, OUTPUT_ROOT)

print("\nProcessing testing split ...")
process_split(TEST_SEQUENCES, "test", DATASET_ROOT, OUTPUT_ROOT)

print("\nDone. Landmark files saved to:", OUTPUT_ROOT)